<a href="https://colab.research.google.com/github/angioitoan2409/flood_forecasting/blob/main/test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os

base = "/content/drive/MyDrive/flood_forecasting/model_inputs"
print("Does the folder exist?", os.path.exists(base))

if os.path.exists(base):
    print("Contents:")
    for f in os.listdir(base):
        print(" ", f)
else:
    # walk up one level to see what IS there
    parent = "/content/drive/MyDrive/flood_forecasting"
    print("Checking parent instead:", parent)
    if os.path.exists(parent):
        print("Contents of parent:")
        for f in os.listdir(parent):
            print(" ", f)
    else:
        print("Parent doesn't exist either - check top-level MyDrive")
        print(os.listdir("/content/drive/MyDrive"))

Does the folder exist? True
Contents:
  dtm.tif
  sealed_fraction.tif
  buildings.tif
  nonsealed_fraction.tif
  manning_n.tif
  radolan_events


In [3]:
%%writefile rim2d_lite_stage1.py

"""
rim2d_lite_stage1.py

STAGE 1: bare local-inertia surface-flow solver.
No rainfall, no drainage yet -- just terrain routing, so we can verify the
core numerics are stable before adding complexity.

This implements the same numerical scheme described in the paper
(Section 2.1): the local inertial approximation (LIA) of the shallow water
equations (Bates, Horritt & Fewtrell, 2010), with the numerical diffusion /
stability scheme of de Almeida & Bates (2013).

Grid convention: NoData cells (buildings + outside sealing-data domain) are
treated as walls -- water cannot enter or leave them, exactly matching
RIM2D's own approach of excluding NoData cells from the hydraulic solve.

Run this in Colab (same session as your preprocessing, so dtm/buildings/etc.
are already in memory -- OR load the .tif files fresh, see load_from_tif()).
"""

import numpy as np

G = 9.81          # gravity, m/s^2
DX = 5.0          # cell size, m (matches your 5m grid)
THETA = 0.7       # de Almeida & Bates (2013) weighting/diffusion parameter (0.7-1.0 typical)
ALPHA = 0.7       # CFL-type timestep safety factor (paper-typical value; tune if unstable)


# ---------------------------------------------------------------------------
# 0. Load inputs (if starting a fresh Colab session, read straight from tif)
# ---------------------------------------------------------------------------
def load_from_tif(out_dir):
    import rasterio
    with rasterio.open(f"{out_dir}/dtm.tif") as src:
        z = src.read(1)
        nodata = src.nodata
        transform = src.transform
    with rasterio.open(f"{out_dir}/manning_n.tif") as src:
        n_manning = src.read(1)
        n_nodata = src.nodata

    active = (z != nodata)   # True = valid hydraulic cell, False = wall (building/no-data)

    # Manning's n is NoData in the same area as sealed_fraction (domain-mask gap).
    # Buildings, however, are only NoData in z, not necessarily in manning_n
    # (manning_n was derived from sealed_fraction, which doesn't distinguish
    # buildings). Fill any remaining gaps in n with a reasonable default so
    # the array has no NaNs sitting inside active cells.
    n_manning = np.where(n_manning == n_nodata, 0.035, n_manning)
    n_manning = np.where(active, n_manning, 0.035)  # value doesn't matter outside domain

    z_filled = np.where(active, z, np.nanmax(z[active]) + 100)  # push walls "high" so they never look like a downhill

    return z_filled, n_manning, active, transform


# ---------------------------------------------------------------------------
# 1. Flux update (local inertial approximation, de Almeida & Bates 2013 form)
# ---------------------------------------------------------------------------
def update_flux_1d(q_prev, z_a, z_b, h_a, h_b, n_face, dt, dx):
    """
    Compute updated discharge per unit width q [m^2/s] across a face between
    two adjacent cells a -> b.

    q_prev : discharge from the previous timestep across this face
    z_a,z_b: bed elevation of the two cells
    h_a,h_b: water depth of the two cells
    n_face : Manning's n to use for this face (e.g. average of the two cells)
    """
    wse_a = z_a + h_a   # water surface elevation
    wse_b = z_b + h_b

    # "Flow depth" hflow: de Almeida/Bates convention -- max WSE minus max bed
    # elevation of the two cells. This is what actually drives the flow,
    # avoiding negative/undefined depths at partially-wet faces.
    hflow = np.maximum(wse_a, wse_b) - np.maximum(z_a, z_b)
    hflow = np.maximum(hflow, 0.0)

    slope = (wse_a - wse_b) / dx   # positive = flow from a to b

    # Avoid division by zero where there's no water at the face at all
    wet = hflow > 1e-6

    denom = 1.0 + G * dt * (n_face**2) * np.abs(q_prev) / np.where(
        wet, hflow**(7.0/3.0), 1.0
    )

    q_new = (THETA * q_prev + (1 - THETA) * 0  # simple theta-averaging term (can extend to neighbor avg)
             + G * hflow * dt * slope) / denom

    q_new = np.where(wet, q_new, 0.0)
    return q_new


# ---------------------------------------------------------------------------
# 2. Adaptive timestep (CFL-type criterion, Hunter et al. / de Almeida et al.)
# ---------------------------------------------------------------------------
def compute_adaptive_dt(h, dx, alpha=ALPHA, dt_min=0.01, dt_max=30.0):
    h_max = np.max(h)
    if h_max <= 1e-6:
        return dt_max
    dt = alpha * dx / np.sqrt(G * h_max)
    return float(np.clip(dt, dt_min, dt_max))


# ---------------------------------------------------------------------------
# 3. One full timestep: update qx, qy, then update h from continuity
# ---------------------------------------------------------------------------
def step(z, h, qx, qy, n_manning, active, dx=DX, dt=None):
    if dt is None:
        dt = compute_adaptive_dt(h, dx)

    ny, nx = z.shape

    # --- x-direction faces (between column j and j+1) ---
    z_a = z[:, :-1]; z_b = z[:, 1:]
    h_a = h[:, :-1]; h_b = h[:, 1:]
    n_face_x = 0.5 * (n_manning[:, :-1] + n_manning[:, 1:])
    face_active_x = active[:, :-1] & active[:, 1:]

    qx_new = update_flux_1d(qx, z_a, z_b, h_a, h_b, n_face_x, dt, dx)
    qx_new = np.where(face_active_x, qx_new, 0.0)  # walls: no flow across

    # --- y-direction faces (between row i and i+1) ---
    z_a = z[:-1, :]; z_b = z[1:, :]
    h_a = h[:-1, :]; h_b = h[1:, :]
    n_face_y = 0.5 * (n_manning[:-1, :] + n_manning[1:, :])
    face_active_y = active[:-1, :] & active[1:, :]

    qy_new = update_flux_1d(qy, z_a, z_b, h_a, h_b, n_face_y, dt, dx)
    qy_new = np.where(face_active_y, qy_new, 0.0)

    # --- continuity: update depth from net flux divergence ---
    dh = np.zeros_like(h)
    # x-direction contributes: inflow from left face - outflow to right face
    dh[:, 1:]  += qx_new * dt / dx   # gains water flowing in from the left
    dh[:, :-1] -= qx_new * dt / dx   # loses water flowing out to the right
    dh[1:, :]  += qy_new * dt / dx
    dh[:-1, :] -= qy_new * dt / dx

    h_new = h + dh
    h_new = np.where(active, h_new, 0.0)   # walls always stay dry
    h_new = np.maximum(h_new, 0.0)          # no negative depths

    return h_new, qx_new, qy_new, dt


# ---------------------------------------------------------------------------
# 4. Simple test harness: drop a pulse of water and watch it flow downhill
# ---------------------------------------------------------------------------
def run_test_pulse(z, n_manning, active, n_steps=200, pulse_depth=0.3, pulse_size=10):
    h = np.zeros_like(z)
    qx = np.zeros((z.shape[0], z.shape[1]-1))
    qy = np.zeros((z.shape[0]-1, z.shape[1]))

    # Drop a pulse near the center of the active domain
    active_rows, active_cols = np.where(active)
    cy, cx = int(np.mean(active_rows)), int(np.mean(active_cols))
    h[cy-pulse_size:cy+pulse_size, cx-pulse_size:cx+pulse_size] = pulse_depth
    h = np.where(active, h, 0.0)

    total_time = 0.0
    for step_i in range(n_steps):
        h, qx, qy, dt = step(z, h, qx, qy, n_manning, active)
        total_time += dt
        if step_i % 20 == 0:
            print(f"step {step_i:4d}  dt={dt:6.3f}s  t={total_time:8.1f}s  "
                  f"max_h={h.max():.4f}m  total_vol={h.sum()*DX*DX:.1f} m3")

    return h, qx, qy


if __name__ == "__main__":
    # Example standalone usage (edit path):
    z, n_manning, active, transform = load_from_tif("/content/drive/MyDrive/flood_forecasting/model_inputs")
    print("Grid:", z.shape, " active cells:", active.sum())
    h_final, qx_final, qy_final = run_test_pulse(z, n_manning, active)

Overwriting rim2d_lite_stage1.py


In [4]:
%%writefile rim2d_lite_stage2.py

"""
rim2d_lite_stage2.py

STAGE 2: adds rainfall forcing + capacity-based drainage (Section 2.2) on
top of the Stage 1 local-inertia surface-flow solver.

Drainage scheme (matching the paper):
  - Building cells: already excluded from routing (walls) in Stage 1 -- this
    is mechanically equivalent to "unlimited roof drainage", since rain
    never accumulates there either way.
  - Sealed surface: sewer drainage removes water at `sewer_capacity` [mm/h],
    but ONLY when depth exceeds `sewer_threshold` [mm]. Below threshold,
    drainage is zero (Section 2.2).
  - Non-sealed surface: infiltration removes water at `infiltration_rate`
    [mm/h], continuously (no threshold mentioned for infiltration in the
    paper).

Rainfall forcing: RADOLAN data is 5-min buckets; the solver's own adaptive
timestep is much finer, so each physics sub-step gets a slice of the current
5-min bucket's rainfall (rate-based, not dumped all at once).
"""

import numpy as np
import pickle
from datetime import timedelta

from rim2d_lite_stage1 import (
    G, DX, load_from_tif, update_flux_1d, compute_adaptive_dt, step as stage1_step
)

# ---------------------------------------------------------------------------
# Drainage parameters (Table 2 in the paper -- reuse as first approximation)
# ---------------------------------------------------------------------------
INFILTRATION_RATE_MMH = 10.0   # mm/h, Section 2.4.4
SEWER_CAPACITY_MMH = 20.0      # mm/h, Table 2 model run 1 (or 25.0 for run 2)
SEWER_THRESHOLD_MM = 5.0       # mm, Table 2 model run 1 (or 2.0 for runs 3/4)


# ---------------------------------------------------------------------------
# Rainfall lookup: given simulation time, find + reproject the right RADOLAN bucket
# ---------------------------------------------------------------------------
class RainfallForcing:
    def __init__(self, pickle_path, out_profile):
        with open(pickle_path, "rb") as f:
            data = pickle.load(f)
        self.event_start = data["event_start"]
        self.event_end = data["event_end"]
        self.timestamps = data["timestamps"]
        self.compact_data = data["compact_data"]
        self.out_profile = out_profile
        self._cache_idx = None
        self._cache_grid = None

    def get_rain_rate_mmh(self, sim_time):
        """
        Returns a (height, width) array of rainfall RATE in mm/h for whichever
        5-min bucket sim_time falls into. Returns zeros if sim_time is outside
        the event window entirely.
        """
        if sim_time < self.timestamps[0] or sim_time > self.timestamps[-1]:
            return np.zeros((self.out_profile["height"], self.out_profile["width"]), dtype="float32")

        # Find the bucket: each timestamp represents the 5 min ENDING at that time
        # (RADOLAN YW convention: file YYYYMMDD_HHMM covers the 5 min up to HHMM)
        idx = np.searchsorted([t for t in self.timestamps], sim_time, side="left")
        idx = min(idx, len(self.timestamps) - 1)

        if idx != self._cache_idx:
            sub, xll, yll, cellsize, nodata = self.compact_data[idx]
            # local import to avoid making stage1 depend on rasterio.warp at module load
            import rasterio
            from rasterio.transform import from_origin
            from rasterio.warp import reproject, Resampling
            from rasterio.crs import CRS

            RADOLAN_CRS_LOCAL = CRS.from_proj4(
                "+proj=stere +lat_0=90 +lat_ts=60 +lon_0=10 "
                "+a=6370040 +b=6370040 +x_0=0 +y_0=0 +units=m +no_defs"
            )
            nrows_sub = sub.shape[0]
            src_transform = from_origin(xll, yll + nrows_sub * cellsize, cellsize, cellsize)

            dst_arr = np.zeros((self.out_profile["height"], self.out_profile["width"]), dtype="float32")
            reproject(
                source=sub,
                destination=dst_arr,
                src_transform=src_transform,
                src_crs=RADOLAN_CRS_LOCAL,
                src_nodata=nodata,
                dst_transform=self.out_profile["transform"],
                dst_crs=self.out_profile["crs"],
                dst_nodata=0.0,
                resampling=Resampling.bilinear,
            )
            dst_arr = np.maximum(dst_arr, 0.0)

            # Convert mm/5min -> mm/h rate
            rate_mmh = dst_arr * 12.0

            self._cache_idx = idx
            self._cache_grid = rate_mmh

        return self._cache_grid


# ---------------------------------------------------------------------------
# Drainage application (Section 2.2)
# ---------------------------------------------------------------------------
def apply_rainfall_and_drainage(h, rain_rate_mmh, sealed_frac, nonsealed_frac, active, dt,
                                  infiltration_rate_mmh=INFILTRATION_RATE_MMH,
                                  sewer_capacity_mmh=SEWER_CAPACITY_MMH,
                                  sewer_threshold_mm=SEWER_THRESHOLD_MM):
    """
    Adds rainfall depth, then removes drainage (sewer on sealed fraction,
    infiltration on non-sealed fraction), for this physics sub-step of
    duration dt [s]. Returns (h_new, sewer_removed_vol, infil_removed_vol)
    -- volumes in m^3, for building the output discharge time series later.
    """
    dt_hours = dt / 3600.0

    # 1. Add rainfall (convert mm/h rate -> m depth over dt)
    rain_depth_m = (rain_rate_mmh / 1000.0) * dt_hours
    h_new = h + np.where(active, rain_depth_m, 0.0)

    # 2. Sewer drainage on sealed fraction, only where depth exceeds threshold
    threshold_m = sewer_threshold_mm / 1000.0
    sewer_capacity_depth_m = (sewer_capacity_mmh / 1000.0) * dt_hours
    above_threshold = h_new > threshold_m
    sewer_removed_depth = np.where(above_threshold, sewer_capacity_depth_m, 0.0) * sealed_frac
    sewer_removed_depth = np.minimum(sewer_removed_depth, np.maximum(h_new - 0, 0))  # can't remove more than present

    # 3. Infiltration on non-sealed fraction, no threshold
    infil_capacity_depth_m = (infiltration_rate_mmh / 1000.0) * dt_hours
    infil_removed_depth = infil_capacity_depth_m * nonsealed_frac
    infil_removed_depth = np.minimum(infil_removed_depth,
                                       np.maximum(h_new - sewer_removed_depth, 0))

    total_removed_depth = sewer_removed_depth + infil_removed_depth
    h_new = h_new - np.where(active, total_removed_depth, 0.0)
    h_new = np.maximum(h_new, 0.0)

    cell_area = DX * DX
    sewer_removed_vol = np.sum(np.where(active, sewer_removed_depth, 0.0)) * cell_area
    infil_removed_vol = np.sum(np.where(active, infil_removed_depth, 0.0)) * cell_area

    return h_new, sewer_removed_vol, infil_removed_vol


# ---------------------------------------------------------------------------
# Full event simulation loop
# ---------------------------------------------------------------------------
def run_event_simulation(z, n_manning, active, sealed_frac, nonsealed_frac,
                          rainfall: RainfallForcing, event_start, event_end,
                          output_interval_s=300, max_steps=200000):
    """
    Runs the solver from event_start to event_end, applying rainfall +
    drainage each physics sub-step. Returns a time series of:
      - sim_times (datetime list, every output_interval_s)
      - sewer_discharge_series (m^3 drained by sewer, per output interval)
      - total_ponded_volume (m^3 of water still sitting on the surface)
    """
    h = np.zeros_like(z)
    qx = np.zeros((z.shape[0], z.shape[1]-1))
    qy = np.zeros((z.shape[0]-1, z.shape[1]))

    sim_time = event_start
    elapsed_since_output = 0.0
    accumulated_sewer_vol = 0.0

    sim_times = [sim_time]
    sewer_discharge_series = [0.0]
    ponded_volume_series = [0.0]

    step_count = 0
    while sim_time < event_end and step_count < max_steps:
        # Compute ONE dt per iteration, used consistently for both rainfall
        # application and flow routing (fixes the rainfall/routing timestep
        # mismatch from the earlier draft).
        dt = compute_adaptive_dt(h, DX)
        remaining = (event_end - sim_time).total_seconds()
        dt = min(dt, remaining) if remaining > 0 else dt

        rain_rate = rainfall.get_rain_rate_mmh(sim_time)
        h, sewer_vol, infil_vol = apply_rainfall_and_drainage(
            h, rain_rate, sealed_frac, nonsealed_frac, active, dt
        )

        h, qx, qy, dt_used = stage1_step(z, h, qx, qy, n_manning, active, dx=DX, dt=dt)
        assert dt_used == dt  # sanity: confirm the same dt was actually used

        accumulated_sewer_vol += sewer_vol
        elapsed_since_output += dt_used
        sim_time += timedelta(seconds=dt_used)
        step_count += 1

        # Heartbeat: print every N raw physics steps, regardless of output
        # interval timing, so we always get visible progress feedback.
        if step_count % 200 == 0:
            pct = 100 * (sim_time - event_start).total_seconds() / (event_end - event_start).total_seconds()
            print(f"  [heartbeat] step={step_count}  sim_time={sim_time}  "
                  f"({pct:.1f}% of event)  dt={dt_used:.3f}s  max_h={h.max():.4f}m")

        if elapsed_since_output >= output_interval_s:
            sim_times.append(sim_time)
            sewer_discharge_series.append(accumulated_sewer_vol)
            ponded_volume_series.append(np.sum(np.where(active, h, 0.0)) * DX * DX)
            accumulated_sewer_vol = 0.0
            elapsed_since_output = 0.0

    print(f"Simulation done: {step_count} physics steps, "
          f"{len(sim_times)} output intervals")
    return sim_times, sewer_discharge_series, ponded_volume_series

Overwriting rim2d_lite_stage2.py


In [5]:
import numpy as np
import rasterio
from datetime import datetime

from rim2d_lite_stage1 import load_from_tif
from rim2d_lite_stage2 import RainfallForcing, run_event_simulation
import rasterio
from datetime import datetime

OUT_DIR = "/content/drive/MyDrive/flood_forecasting/model_inputs"

# Load terrain/roughness (Stage 1)
z, n_manning, active, transform = load_from_tif(OUT_DIR)

# Load sealed/non-sealed fraction rasters too (needed for drainage)
with rasterio.open(f"{OUT_DIR}/sealed_fraction.tif") as src:
    sealed_frac = src.read(1)
    sealed_nodata = src.nodata
with rasterio.open(f"{OUT_DIR}/nonsealed_fraction.tif") as src:
    nonsealed_frac = src.read(1)

sealed_frac = np.where(sealed_frac == sealed_nodata, 0.0, sealed_frac)
nonsealed_frac = np.where(nonsealed_frac == sealed_nodata, 1.0, nonsealed_frac)

# Load rainfall for the 2021-07-16 event (smallest of the 8 to test first...
# actually 2017-07-09 is smallest at 41 steps, good first test)
rainfall = RainfallForcing(f"{OUT_DIR}/radolan_events/2017-07-09_radolan.pkl",
                             {"height": z.shape[0], "width": z.shape[1],
                              "transform": transform, "crs": "EPSG:25833", "nodata": -9999.0})

event_start = datetime(2017, 7, 9, 17, 55)
event_end = datetime(2017, 7, 9, 21, 15)

sim_times, sewer_discharge, ponded_vol = run_event_simulation(
    z, n_manning, active, sealed_frac, nonsealed_frac,
    rainfall, event_start, event_end
)

  [heartbeat] step=200  sim_time=2017-07-09 17:58:16.601229  (1.6% of event)  dt=0.779s  max_h=2.0584m
  [heartbeat] step=400  sim_time=2017-07-09 18:00:44.914879  (2.9% of event)  dt=0.715s  max_h=2.4435m
  [heartbeat] step=600  sim_time=2017-07-09 18:03:05.034391  (4.0% of event)  dt=0.689s  max_h=2.6312m
  [heartbeat] step=800  sim_time=2017-07-09 18:05:21.096105  (5.2% of event)  dt=0.672s  max_h=2.7624m
  [heartbeat] step=1000  sim_time=2017-07-09 18:07:34.226667  (6.3% of event)  dt=0.660s  max_h=2.8714m
  [heartbeat] step=1200  sim_time=2017-07-09 18:09:45.087938  (7.4% of event)  dt=0.650s  max_h=2.9597m


KeyboardInterrupt: 